In [0]:
# Master Import Cell
from pyspark.sql.functions import col, when, udf, length
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Confirming imports are loaded
print("All Spark functions and ML libraries imported successfully.")


In [0]:
# Install NLP library
%pip install textblob

# Restart Python to register the new library
dbutils.library.restartPython()

# Standard Imports
from pyspark.sql.functions import col, length, when, udf
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from textblob import TextBlob


In [0]:
from pyspark.sql.functions import col, length, get_json_object

# Defining Production Paths
reviews_path = "/Volumes/workspace/default/project/Toys_and_Games.jsonl.gz"
meta_path = "/Volumes/workspace/default/project/meta_Toys_and_Games.jsonl.gz"

# 2. Ingesting Reviews
df_reviews = spark.read.json(reviews_path)

# 3. Robust Metadata Ingestion (To Handle the duplicate column error)
df_meta_raw = spark.read.text(meta_path)

df_meta = df_meta_raw.select(
    get_json_object(col("value"), "$.parent_asin").alias("parent_asin"),
    get_json_object(col("value"), "$.main_category").alias("meta_main_category"),
    get_json_object(col("value"), "$.price").alias("meta_price"),
    get_json_object(col("value"), "$.title").alias("meta_title")
).dropna(subset=["parent_asin"])

# 4. Data Profile: Record Counts
print(f"Total Review Records: {df_reviews.count()}")
print(f"Total Metadata Records: {df_meta.count()}")

# 5. Exploratory Data Analysis
df_reviews = df_reviews.withColumn("review_length", length(col("text")))

print("--- Rating Distribution ---")
display(df_reviews.groupBy("rating").count().orderBy("rating"))

print("--- Review Length Distribution ---")
display(df_reviews.select("review_length"))




In [0]:
from pyspark.sql.functions import col, when

# Distributed Join
# We used an inner join on 'parent_asin' to ensure we only keep reviews with matching product info
df_joined = df_reviews.join(df_meta, on="parent_asin", how="inner")

# Data Cleaning
# Standardizing 'price' to a numeric format and defining our target 'label'
# A rating > 4 is defined as a 'Success' (1), otherwise (0)
df_prepared = df_joined.withColumn("price", col("meta_price").cast("float")) \
                       .withColumn("label", when(col("rating") > 4, 1).otherwise(0))

# Handling Missing Values
# Removing any rows that lack essential data to prevent errors during modeling
df_clean = df_prepared.dropna(subset=["price", "text", "meta_main_category", "rating"])

print("Phase 3: Data Preparation Complete.")
print(f"Final Record Count for Modeling: {df_clean.count()}")

In [0]:
from textblob import TextBlob
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType

# Defining Sentiment Logic
def get_sentiment(text):
    if text is None:
        return 0.0
    return TextBlob(text).sentiment.polarity

# Registering the User Defined Function (UDF)
sentiment_udf = udf(get_sentiment, FloatType())

# Applying Sentiment Analysis to the 11.2M records
df_analytics = df_clean.withColumn("sentiment_score", sentiment_udf(col("text")))

# Statistical Validation: Pearson Correlation
correlation = df_analytics.stat.corr("sentiment_score", "rating")

print(f"Phase 4: Analytical Depth Complete.")
print(f"Pearson Correlation (Sentiment vs Rating): {correlation:.4f}")

# Preview results
display(df_analytics.select("text", "rating", "sentiment_score").limit(10))

In [0]:
# Product Performance Analysis

from pyspark.sql.functions import avg, count

performance_analysis = df_clean.groupBy("meta_main_category").agg(
    avg("helpful_vote").alias("avg_helpfulness"),
    avg("rating").alias("avg_rating"),
    count("rating").alias("review_count")
).orderBy("avg_helpfulness", ascending=False)

print("Phase 4.5: Product Performance Analysis Complete.")
# Showing the top categories where reviews are most influential
display(performance_analysis)

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Encode Categorical Data
indexer = StringIndexer(inputCol="meta_main_category", outputCol="category_index", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="category_index", outputCol="category_vec")

#Vector Assembly
assembler = VectorAssembler(
    inputCols=["sentiment_score", "price", "category_vec"], 
    outputCol="features"
)

# Transform the Data
df_indexed = indexer.fit(df_analytics).transform(df_analytics)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)
ml_data = assembler.transform(df_encoded)

print("Phase 5: Feature Engineering Complete.")
# Preview the features vector
display(ml_data.select("features", "label").limit(5))

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Train/Test Split
train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=42)

# Model Training
# Initializing the Logistic Regression 
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)

# Predictions
predictions = lr_model.transform(test_data)

# Evaluation: F1-Score
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator.evaluate(predictions)

print(f"Final Project Evaluation - F1-Score: {f1_score:.4f}")

# Confusion Matrix
print("--- Confusion Matrix (Actual vs. Predicted) ---")
predictions.crosstab("prediction", "label").show()

In [0]:
# Phase 7: Deployment - Clean Export
export_path = "/Volumes/workspace/default/project/toy_analysis_results.csv"

# Sampling 10,000 rows
df_export = predictions.select(
    "text", 
    "rating", 
    "sentiment_score", 
    "prediction", 
    "label", 
    "meta_main_category"
).limit(10000).toPandas()

df_export.to_csv(export_path, index=False)
print(f"Deployment successful! Exported to: {export_path}")